# Factual RAG evaluation: real responses

Run the actual AniList RAG against `questions.csv`, extract assertions from its final natural-language
responses using a separate structured Ollama call, and compare those values with frozen database labels.

The benchmark contains **42 questions across six titles and seven fields**. The extraction call sees
only the question, requested field and final answer; reference values are withheld. It must preserve
wrong assertions and provide verbatim supporting quotes. Automatic extraction is still fallible:
audit quotes and manually review uncertain results. Using a separate stronger installed extractor
model can reduce correlated mistakes; the default uses the configured local model.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/factual_evaluation.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter inside this repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_settings
from src.factual_evaluation import load_questions, run_evaluation, read_csv, score_values, summarize

DATA = ROOT / 'data/evaluation/factual'
QUESTIONS_CSV = DATA / 'questions.csv'
SETTINGS = load_settings(ROOT / 'settings.toml', 'anilist')
DATABASE = SETTINGS.documents / 'catalog.sqlite3'
questions = load_questions(QUESTIONS_CSV, DATABASE)
print(f'{len(questions)} questions; RAG model: {SETTINGS.llm}')
print('Fields:', sorted({q['field'] for q in questions}))


## Run the real pipeline

Ollama must be running with the configured model available, and AniList setup must already exist.
This cell performs real model inference and can take several minutes. Each question starts with empty
history and fresh agent state, while sharing loaded models. No expected values are passed to the RAG.
The separate extraction request uses temperature zero; the RAG keeps your generation settings.

Set `LIMIT=3` for a smoke test or `None` for every question. Use a different `RUN_NAME` when changing
the limit, settings, question CSV or extractor. Identical runs resume per-question checkpoints,
including failed rows; use a new run directory for retries or repeated measurements.


In [ ]:
RUN_NAME = 'real_model_v1'
LIMIT = None
EXTRACTOR_MODEL = None  # None uses SETTINGS.llm; otherwise an installed Ollama model name.
OUTPUT = DATA / 'results' / RUN_NAME

report = run_evaluation(SETTINGS, QUESTIONS_CSV, OUTPUT,
                        extractor_model=EXTRACTOR_MODEL, limit=LIMIT)
print(json.dumps(report['overall'], indent=2))


## Metrics and failure analysis

- **End-to-end exact accuracy:** completely correct answers / all attempted questions. Inference errors,
  uncertain extraction, and extraction errors count as unsuccessful attempts here.
- **Scoring coverage:** fraction automatically scored. Always inspect it alongside accuracy.
- **Scored exact accuracy:** exact correctness among successfully interpreted responses, including unanswered failures.
- **Macro precision / recall / F1:** field-value metrics among scored questions, giving each question equal weight.
  Extra incorrect values hurt precision; missing values hurt recall. Unanswered responses score zero.

The scorer normalizes capitalization, underscores, numeric formatting and explicit reviewed aliases.
It does not use fuzzy similarity to award credit. Scores concern only the requested field, not every
claim in the response. Database tag/role choices define this benchmark's truth.


In [ ]:
print(json.dumps(report['by_field'], indent=2))
answers = read_csv(OUTPUT / 'answers.csv')
for row in answers:
    if row['exact_match'] != '1':
        print(row['question_id'], row['question'])
        print('  Status:', row['status'], '| Answer:', row['answer_text'])
        print('  Expected:', row['expected_values'], '| Extracted:', row['extracted_values'])
        print('  Evidence:', row['extraction'])
        if row['error']:
            print('  Error:', row['error'])
print('Full answers:', OUTPUT / 'answers.csv')
print('Run settings:', OUTPUT / 'manifest.json')


## Optional: offline scorer smoke test

The original correct/mixed CSVs remain **synthetic fixtures**. They test deterministic scoring without
calling a model. The template remains available for human annotation and extractor validation.
The 42-row mixed fixture has six intentionally incorrect/incomplete/unanswered rows (85.7% exact accuracy).


In [ ]:
fixture = read_csv(DATA / 'answers_mixed.csv')  # or answers_correct.csv
question_map = {q['question_id']: q for q in questions}
fixture_rows = []
for answer in fixture:
    if answer['reviewed'] != 'true':
        continue
    question = question_map[answer['question_id']]
    fixture_rows.append({**question, 'status': answer['answer_status'], 'model_seconds': 0,
                         'extraction_seconds': 0,
                         'metrics': score_values(json.loads(question['expected_values']),
                                                 json.loads(answer['extracted_values']), question['field'])})
print(json.dumps(summarize(fixture_rows), indent=2))


## Artifacts and interpretation

`answers.csv` preserves final prose, extracted values, quotes, metrics, timings and errors.
Each question's JSON checkpoint additionally saves RAG usage and debug events. `summary.json` reports
overall and per-field metrics. `manifest.json` locks question-file hash, settings and extractor choice.
Model startup is included in the first question's timing; extraction time is recorded separately.

Reference labels are checked against read-only database queries before inference. Catalog drift stops
the run and requires intentional benchmark versioning. Studio labels require both main and animation
flags; character labels include every role MAIN. This small sample is not a representative quality estimate.
Human-audit extracted values before treating automated accuracy as a reliable model-quality measurement.
